In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.makedirs('/content/drive/MyDrive/BhramGuard/datasets', exist_ok=True)

In [3]:
!pip install -q huggingface_hub

In [4]:
from huggingface_hub import hf_hub_download

# List of all dataset files we found earlier
filenames = ["texts.json", "urls.json", "webs.json", "combined_full.json", "combined_reduced.json"]

downloaded_paths = {}

for fname in filenames:
    path = hf_hub_download(
        repo_id="ealvaradob/phishing-dataset",
        filename=fname,
        repo_type="dataset"
    )
    downloaded_paths[fname] = path
    print(f"{fname} -> {path}")

texts.json:   0%|          | 0.00/52.1M [00:00<?, ?B/s]

texts.json -> /root/.cache/huggingface/hub/datasets--ealvaradob--phishing-dataset/snapshots/55caf9af01ae498d68ff44886ff69a4b165eb54b/texts.json


urls.json:   0%|          | 0.00/73.2M [00:00<?, ?B/s]

urls.json -> /root/.cache/huggingface/hub/datasets--ealvaradob--phishing-dataset/snapshots/55caf9af01ae498d68ff44886ff69a4b165eb54b/urls.json


webs.json:   0%|          | 0.00/465M [00:00<?, ?B/s]

webs.json -> /root/.cache/huggingface/hub/datasets--ealvaradob--phishing-dataset/snapshots/55caf9af01ae498d68ff44886ff69a4b165eb54b/webs.json


combined_full.json:   0%|          | 0.00/591M [00:00<?, ?B/s]

combined_full.json -> /root/.cache/huggingface/hub/datasets--ealvaradob--phishing-dataset/snapshots/55caf9af01ae498d68ff44886ff69a4b165eb54b/combined_full.json


combined_reduced.json:   0%|          | 0.00/521M [00:00<?, ?B/s]

combined_reduced.json -> /root/.cache/huggingface/hub/datasets--ealvaradob--phishing-dataset/snapshots/55caf9af01ae498d68ff44886ff69a4b165eb54b/combined_reduced.json


In [9]:
import json
import pandas as pd

dataframes = {}

for fname, path in downloaded_paths.items():
    with open(path, 'r') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    dataframes[fname] = df
    print(f"\n{fname}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5))


texts.json
Shape: (20137, 2)
Columns: ['text', 'label']
                                                text  label
0  re : 6 . 1100 , disc : uniformitarianism , re ...      0
1  the other side of * galicismos * * galicismo *...      0
2  re : equistar deal tickets are you still avail...      0
3  \nHello I am your hot lil horny toy.\n    I am...      1
4  software at incredibly low prices ( 86 % lower...      1

urls.json
Shape: (835697, 2)
Columns: ['text', 'label']
                                                text  label
0      http://webmail-brinkster.com/ex/?email=%20%0%      1
1                         billsportsmaps.com/?p=1206      0
2  www.sanelyurdu.com/language/homebank.tsbbank.c...      1
3                          ee-billing.limited323.com      1
4                   indiadaily.com/bolly_archive.htm      0

webs.json
Shape: (15756, 2)
Columns: ['text', 'label']
                                                text  label
0  <!doctypehtml PUBLIC "-//W3C//DTD HTML 4.01 Tr.

In [10]:
print("texts.json label distribution:")
print(dataframes["texts.json"]["label"].value_counts())
print(dataframes["texts.json"]["label"].value_counts(normalize=True))

print("\nurls.json label distribution:")
print(dataframes["urls.json"]["label"].value_counts())
print(dataframes["urls.json"]["label"].value_counts(normalize=True))

texts.json label distribution:
label
0    12465
1     7672
Name: count, dtype: int64
label
0    0.61901
1    0.38099
Name: proportion, dtype: float64

urls.json label distribution:
label
0    444933
1    390764
Name: count, dtype: int64
label
0    0.532409
1    0.467591
Name: proportion, dtype: float64


In [11]:
dataframes["texts.json"]["text_length"] = dataframes["texts.json"]["text"].str.len()
print(dataframes["texts.json"]["text_length"].describe())

count    2.013700e+04
mean     2.505063e+03
std      1.201156e+05
min      1.000000e+00
25%      2.810000e+02
50%      7.440000e+02
75%      1.682000e+03
max      1.703669e+07
Name: text_length, dtype: float64


In [12]:
dataframes["texts.json"].to_csv('/content/drive/MyDrive/BhramGuard/datasets/texts.csv', index=False)
dataframes["urls.json"].to_csv('/content/drive/MyDrive/BhramGuard/datasets/urls.csv', index=False)

print("Saved successfully")

Saved successfully


In [13]:
print("=== LEGITIMATE EXAMPLE ===")
print(dataframes["texts.json"][dataframes["texts.json"]["label"] == 0]["text"].iloc[0])

print("\n=== PHISHING EXAMPLE ===")
print(dataframes["texts.json"][dataframes["texts.json"]["label"] == 1]["text"].iloc[0])

=== LEGITIMATE EXAMPLE ===
re : 6 . 1100 , disc : uniformitarianism , re : 1086 ; sex / lang dick hudson 's observations on us use of 's on ' but not 'd aughter ' as a vocative are very thought-provoking , but i am not sure that it is fair to attribute this to " sons " being " treated like senior relatives " . for one thing , we do n't normally use ' brother ' in this way any more than we do 'd aughter ' , and it is hard to imagine a natural class comprising senior relatives and 's on ' but excluding ' brother ' . for another , there seem to me to be differences here . if i am not imagining a distinction that is not there , it seems to me that the senior relative terms are used in a wider variety of contexts , e . g . , calling out from a distance to get someone 's attention , and hence at the beginning of an utterance , whereas 's on ' seems more natural in utterances like ' yes , son ' , ' hand me that , son ' than in ones like ' son ! ' or ' son , help me ! ' ( although perhaps thes

In [21]:
import re

def extract_features(text):
    features = {}

    urgency_words = ['hurry', 'urgent', 'immediately', 'act now', 'limited time', 'expire', 'act fast']
    features['urgency_count'] = sum(1 for word in urgency_words if word in text.lower())

    words = text.split()
    caps_words = [w for w in words if w.isupper() and len(w) > 1]
    features['caps_ratio'] = len(caps_words) / len(words) if words else 0

    features['exclamation_count'] = text.count('!')
    features['elongation_count'] = len(re.findall(r'([a-zA-Z])\1{3,}', text))
    features['link_count'] = len(re.findall(r'http[s]?://|www\.', text))
    features['text_length'] = len(text)

    # NEW: reward/incentive language (common in scams)
    reward_words = ['free', 'win', 'winner', 'prize', 'discount', 'offer', 'guarantee', 'cash']
    features['reward_count'] = sum(1 for word in reward_words if word in text.lower())

    # NEW: authority/threat language (common in phishing)
    threat_words = ['suspend', 'verify', 'confirm', 'account', 'password', 'security', 'unauthorized']
    features['threat_count'] = sum(1 for word in threat_words if word in text.lower())

    return features

feature_list = df_texts["text"].apply(extract_features)
features_df = pd.DataFrame(feature_list.tolist())
df_features = pd.concat([features_df, df_texts["label"]], axis=1)

print(df_features.groupby("label").mean())
# Test it on both examples
legit_text = dataframes["texts.json"][dataframes["texts.json"]["label"] == 0]["text"].iloc[0]
phish_text = dataframes["texts.json"][dataframes["texts.json"]["label"] == 1]["text"].iloc[0]

print("Legit features:", extract_features(legit_text))
print("Phishing features:", extract_features(phish_text))

       urgency_count  caps_ratio  exclamation_count  elongation_count  \
label                                                                   
0           0.040995    0.012292           1.731247          0.039711   
1           0.127477    0.027662           3.016814          0.100756   

       link_count  text_length  reward_count  threat_count  
label                                                       
0        1.341436  3141.494344      0.481268      0.147934  
1        0.411236  1471.028936      0.974322      0.238530  
Legit features: {'urgency_count': 0, 'caps_ratio': 0.0, 'exclamation_count': 2, 'elongation_count': 0, 'link_count': 0, 'text_length': 1030, 'reward_count': 0, 'threat_count': 0}
Phishing features: {'urgency_count': 1, 'caps_ratio': 0.03125, 'exclamation_count': 1, 'elongation_count': 2, 'link_count': 2, 'text_length': 688, 'reward_count': 1, 'threat_count': 0}


In [22]:
feature_list = df_texts["text"].apply(extract_features)
features_df = pd.DataFrame(feature_list.tolist())
df_features = pd.concat([features_df, df_texts["label"]], axis=1)

print("Average features by class (after fix):")
print(df_features.groupby("label").mean())

Average features by class (after fix):
       urgency_count  caps_ratio  exclamation_count  elongation_count  \
label                                                                   
0           0.040995    0.012292           1.731247          0.039711   
1           0.127477    0.027662           3.016814          0.100756   

       link_count  text_length  reward_count  threat_count  
label                                                       
0        1.341436  3141.494344      0.481268      0.147934  
1        0.411236  1471.028936      0.974322      0.238530  


In [23]:
print("Average features by class:")
print(df_features.groupby("label").mean())

Average features by class:
       urgency_count  caps_ratio  exclamation_count  elongation_count  \
label                                                                   
0           0.040995    0.012292           1.731247          0.039711   
1           0.127477    0.027662           3.016814          0.100756   

       link_count  text_length  reward_count  threat_count  
label                                                       
0        1.341436  3141.494344      0.481268      0.147934  
1        0.411236  1471.028936      0.974322      0.238530  


In [24]:
# Check what's causing high elongation in legit emails
legit_high_elong = df_features[(df_features['label']==0) & (df_features['elongation_count'] > 5)]
print(legit_high_elong.shape)

sample_idx = legit_high_elong.index[0]
print(dataframes["texts.json"].loc[sample_idx, "text"][:500])

(7, 9)
icslp 96 = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = iiii ccccc sssss ll pppppp 999999 666666 ii cc cc ss ss ll pp pp 99 99 66 66 ii cc ss ll pp pp 99 99 66 ii cc sssssss ll pppppp 9999999 6666666 ii cc ss ll pp 99 66 66 ii cc cc ss ss ll pp 99 99 66 66 iiii ccccc sssss lllllll pp 999999 666666 = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Build TF-IDF from raw text (captures actual vocabulary/wording patterns)
tfidf = TfidfVectorizer(max_features=1000, stop_words='english', min_df=5, max_df=0.8)
X_tfidf = tfidf.fit_transform(df_texts["text"])

# Convert your hand-crafted features to sparse format so they can combine with TF-IDF
X_manual_sparse = csr_matrix(features_df.values)

# Combine both feature sets into one matrix
X_combined = hstack([X_tfidf, X_manual_sparse])

print("Combined feature matrix shape:", X_combined.shape)

Combined feature matrix shape: (20137, 1008)


In [30]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

gb_model_v2 = GradientBoostingClassifier(random_state=42)
gb_model_v2.fit(X_train_c, y_train_c)

y_pred_v2 = gb_model_v2.predict(X_test_c)

print(classification_report(y_test_c, y_pred_v2))

              precision    recall  f1-score   support

           0       0.90      0.96      0.93      2493
           1       0.93      0.84      0.88      1535

    accuracy                           0.91      4028
   macro avg       0.92      0.90      0.91      4028
weighted avg       0.91      0.91      0.91      4028



In [34]:
import joblib
os.makedirs('/content/drive/MyDrive/BhramGuard/models', exist_ok=True)

joblib.dump(gb_model_v2, '/content/drive/MyDrive/BhramGuard/models/gb_classifier.pkl')
joblib.dump(tfidf, '/content/drive/MyDrive/BhramGuard/models/tfidf_vectorizer.pkl')

['/content/drive/MyDrive/BhramGuard/models/tfidf_vectorizer.pkl']

In [26]:
# model = LogisticRegression(class_weight='balanced', max_iter=1000)
# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)

# print(classification_report(y_test, y_pred))
# print("\nConfusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.84      0.81      2493
           1       0.70      0.62      0.66      1535

    accuracy                           0.75      4028
   macro avg       0.74      0.73      0.73      4028
weighted avg       0.75      0.75      0.75      4028


Confusion Matrix:
[[2084  409]
 [ 584  951]]


In [27]:
# from sklearn.ensemble import GradientBoostingClassifier

# gb_model = GradientBoostingClassifier(random_state=42)
# gb_model.fit(X_train, y_train)
# y_pred_gb = gb_model.predict(X_test)

# print(classification_report(y_test, y_pred_gb))

              precision    recall  f1-score   support

           0       0.80      0.90      0.85      2493
           1       0.81      0.64      0.71      1535

    accuracy                           0.80      4028
   macro avg       0.80      0.77      0.78      4028
weighted avg       0.80      0.80      0.80      4028



In [35]:
import pandas as pd

df_ai_test = pd.read_csv('/content/drive/MyDrive/BhramGuard/datasets/phishing_legitimate_dataset.csv')
print(df_ai_test.shape)
print(df_ai_test.columns.tolist())
df_ai_test.head()

(50, 2)
['text', 'label']


,text,label
0,Your recent banking session was flagged for un...,1
1,Payroll records show your tax withholding prof...,1
2,We could not complete delivery because the add...,1
3,A new payee was added to your online banking p...,1
4,Your company mailbox storage will be reduced t...,1


In [36]:
# Extract same handcrafted features as training
feature_list_ai = df_ai_test["text"].apply(extract_features)
features_ai_df = pd.DataFrame(feature_list_ai.tolist())

# Reuse the SAME fitted TF-IDF vectorizer (transform, not fit_transform)
X_tfidf_ai = tfidf.transform(df_ai_test["text"])

from scipy.sparse import hstack, csr_matrix
X_manual_ai_sparse = csr_matrix(features_ai_df.values)
X_combined_ai = hstack([X_tfidf_ai, X_manual_ai_sparse])

y_pred_ai = gb_model_v2.predict(X_combined_ai)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(df_ai_test["label"], y_pred_ai))
print("\nConfusion Matrix:")
print(confusion_matrix(df_ai_test["label"], y_pred_ai))

              precision    recall  f1-score   support

           0       0.52      0.96      0.68        25
           1       0.75      0.12      0.21        25

    accuracy                           0.54        50
   macro avg       0.64      0.54      0.44        50
weighted avg       0.64      0.54      0.44        50


Confusion Matrix:
[[24  1]
 [22  3]]
